# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Rank opportunities based on high baseline impression volume, declining position trend, and strong click-through rate (CTR), prioritizing queries where small rank improvements yield the highest potential traffic gain.

Reason Codes:

HIGH_IMPRESSION_LOW_CTR: High visibility but underperforming click rates (opportunity to improve titles/snippets).

POSITION_SLIP: Keywords dropping from top 3 or top 10 positions over recent windows.

STABLE_HIGH_POTENTIAL: Keywords holding position 4–10 with steady volume, prime for targeted optimization.

In [1]:
# Setup reason code thresholds and definitions
REASON_CODES = {
    "HIGH_IMPRESSION_LOW_CTR": "Impressions > 1000 and CTR < 2.0%",
    "POSITION_SLIP": "Position dropped by > 1.5 ranks vs prior window",
    "STABLE_HIGH_POTENTIAL": "Average position between 4.0 and 10.0"
}
print("Reason codes initialized:", REASON_CODES)

Reason codes initialized: {'HIGH_IMPRESSION_LOW_CTR': 'Impressions > 1000 and CTR < 2.0%', 'POSITION_SLIP': 'Position dropped by > 1.5 ranks vs prior window', 'STABLE_HIGH_POTENTIAL': 'Average position between 4.0 and 10.0'}


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Calculate the composite action score combining impression weight, rank gap, and CTR potential, sort the dataset descending by score, and save the final queue to work/outputs/baseline_action_score.csv.

In [6]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. Load dataset directly from GitHub
# ============================================================

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# ============================================================
# 2. Create required columns
# ============================================================

df["impressions"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
).fillna(0)

df["position"] = pd.to_numeric(
    df["avg_position"],
    errors="coerce"
)

df["position"] = df["position"].fillna(
    df["position"].median()
)

df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
).fillna(0)


# ============================================================
# 3. Baseline Action Score
# ============================================================

df["baseline_action_score"] = (
    np.log1p(df["impressions"])
    * (df["position"] - 1)
    * df["ctr"]
)


# ============================================================
# 4. Assign Reason Codes
# ============================================================

def assign_reason(row):

    if (
        row["impressions"] > 1000
        and row["ctr"] < 0.02
    ):
        return "HIGH_IMPRESSION_LOW_CTR"

    elif row["position"] > 10:
        return "POSITION_SLIP"

    else:
        return "STABLE_HIGH_POTENTIAL"


df["reason_code"] = df.apply(
    assign_reason,
    axis=1
)


# ============================================================
# 5. Rank content
# ============================================================

ranked_df = (
    df.sort_values(
        by="baseline_action_score",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. Save output
# ============================================================

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

ranked_df.to_csv(
    output_path,
    index=False
)


# ============================================================
# 7. Display results
# ============================================================

print("\nSUCCESS!")
print(f"Saved {len(ranked_df)} rows to:")
print(output_path)

print("\nTop 10 ranked content:")

print(
    ranked_df[
        [
            "content_id",
            "impressions",
            "ctr",
            "position",
            "baseline_action_score",
            "reason_code"
        ]
    ].head(10)
)

print("\nReason Code Distribution:")

print(
    ranked_df["reason_code"].value_counts()
)

Dataset loaded successfully!
Dataset shape: (30000, 44)

SUCCESS!
Saved 30000 rows to:
work/outputs/baseline_action_score.csv

Top 10 ranked content:
             content_id  impressions     ctr  position  baseline_action_score  \
0  content_4592e7c0daaf            2   50.00      98.0            5328.269600   
1  content_e83ed987f5ce           34   29.41      26.3            2645.438498   
2  content_9a7fe374c900            3   66.67      26.3            2338.333400   
3  content_ead8e65d1fd2            8   12.50      80.6            2186.238454   
4  content_6016b918a48f            1  100.00      30.0            2010.126824   
5  content_0577107d816a            5   20.00      52.8            1856.262810   
6  content_0570b79cbd58           46    8.70      54.0            1775.303059   
7  content_cfa4d9f1bf0a            1  100.00      25.0            1663.553233   
8  content_0b02301e549b           38    7.89      53.0            1503.086072   
9  content_9a9cd696637f            3   3

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

ach item, document:

Action: Recommended operational change (e.g., content refresh, title rewrite, internal linking boost).

Reason Code: Assigned code from Section 1.

Confidence Note: Level of confidence (High/Medium/Low) based on sample size and stability observed.

Failure Condition: What specific factors would make this recommendation incorrect (e.g., seasonal intent shift, brand query

In [8]:
import pandas as pd
import os

# Load ranked results
file_path = "work/outputs/baseline_action_score.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File not found: {file_path}. Run the baseline scoring cell first."
    )

ranked_df = pd.read_csv(file_path)

# Display top 20 queue items
top_20 = ranked_df.head(20)

# Columns available in your dataset
display_columns = [
    "content_id",
    "client_id",
    "impressions",
    "position",
    "ctr",
    "baseline_action_score",
    "reason_code"
]

# Keep only columns that actually exist
available_columns = [
    col for col in display_columns
    if col in top_20.columns
]

print("Top 20 Content Refresh Queue:")
print(top_20[available_columns].to_string(index=False))

Top 20 Content Refresh Queue:
          content_id         client_id  impressions  position    ctr  baseline_action_score   reason_code
content_4592e7c0daaf client_9f14025af0            2      98.0  50.00            5328.269600 POSITION_SLIP
content_e83ed987f5ce client_d4735e3a26           34      26.3  29.41            2645.438498 POSITION_SLIP
content_9a7fe374c900 client_9f14025af0            3      26.3  66.67            2338.333400 POSITION_SLIP
content_ead8e65d1fd2 client_9f14025af0            8      80.6  12.50            2186.238454 POSITION_SLIP
content_6016b918a48f client_d4735e3a26            1      30.0 100.00            2010.126824 POSITION_SLIP
content_0577107d816a client_9f14025af0            5      52.8  20.00            1856.262810 POSITION_SLIP
content_0570b79cbd58 client_9f14025af0           46      54.0   8.70            1775.303059 POSITION_SLIP
content_cfa4d9f1bf0a client_d4735e3a26            1      25.0 100.00            1663.553233 POSITION_SLIP
content_0b02301e

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis: Identify specific top-20 queries that look incorrect upon inspection (e.g., navigational brand terms where rank optimization adds no value, or ultra-broad intent terms with high impressions but low conversion intent).

Leakage Check: Verify that no downstream target variables, outcome flags, or future time-window data were included in the calculation of baseline_action_score. Confirm all metrics rely strictly on historical observation windows.

In [9]:
# Verify no future date fields or product outcome flags are present in scoring features
forbidden_columns = ["future_clicks", "conversion_flag", "target_rank", "is_converted"]
leakage_detected = [col for col in forbidden_columns if col in df.columns]

print("Leakage Check Result:", "FAIL - Found leaked columns: " + str(leakage_detected) if leakage_detected else "PASS - No future leakage detected.")

Leakage Check Result: PASS - No future leakage detected.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.